In [1]:
# ipca_pipeline.py (or paste into a notebook cell)

from __future__ import annotations
import os
import contextlib
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict, Any

import numpy as np
import pandas as pd
!pip install ipca
from ipca import InstrumentedPCA

def run_pipeline(
    start_yyyymm: str = "199001",
    end_yyyymm: str = "202412",
    max_nan_frac: float = 0.6,
    cutoff: str = "2015-12-31",
    n_factors: int = 3,
) -> Dict[str, Any]:
    df_raw = download_openap_data(start_yyyymm, end_yyyymm)
    df_keep, dropped = remove_mostly_nan_columns(df_raw, max_nan_frac=max_nan_frac)
    df_filled = fill_remaining_missing(df_keep)

    panel = build_ipca_panel(df_filled, ret_col="excess_ret", shift_target=True)
    train_df, test_df = split_train_test_by_date(panel.df, cutoff=cutoff)

    X_train = train_df[panel.char_cols].to_numpy(np.float64)
    y_train = train_df["y_ipca"].to_numpy(np.float64)
    idx_train = train_df[["i_idx", "t_idx"]].to_numpy(np.int64)

    X_test = test_df[panel.char_cols].to_numpy(np.float64)
    y_test = test_df["y_ipca"].to_numpy(np.float64)
    idx_test = test_df[["i_idx", "t_idx"]].to_numpy(np.int64)

    mod = fit_ipca(X_train, y_train, idx_train, n_factors=n_factors, silent=True)
    r2_train = score_ipca(mod, X_train, y_train, idx_train, mean_factor=False)
    r2_test = score_ipca(mod, X_test, y_test, idx_test, mean_factor=True)  # unseen dates

    return {
        "model": mod,
        "dropped_cols": dropped,
        "char_cols": panel.char_cols,
        "r2_train": r2_train,
        "r2_test_mean_factor": r2_test,
        "train_shape": X_train.shape,
        "test_shape": X_test.shape,
    }


  Preparing metadata (setup.py) ... done
  Created wheel for progressbar: filename=progressbar-2.5-py3-none-any.whl size=12065 sha256=f993ca312161d20fe5d1cc237f5b1f99799ba68fb46b2ae7d210f77248134b4a
  Stored in directory: /root/.cache/pip/wheels/a5/4d/c7/f3cf0f75c746c219090060131fe00f1523cc2c5484991f4030
Successfully built progressbar


In [3]:
import sys
sys.path.append('/content/TheVirtueOfComplexity_PaperReplication_Experimentation-master')
from src.ipca_workflow import IPCAWorkflow

In [6]:
new_mod = IPCAWorkflow()
new_mod.download_openap_data(start_yyyymm="199001", end_yyyymm="202601")

Enter your WRDS username [root]:arshdeep1111
Enter your password:··········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: n
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done

Data is downloaded: 3 mins


,permno,yyyymm,AM,AOP,AbnormalAccruals,Accruals,AccrualsBM,Activism1,Activism2,AdExp,...,sinAlgo,skew1,std_turn,tang,zerotrade12M,zerotrade1M,zerotrade6M,Price,Size,STreversal
0,10001,1990-01-01,1.827961,NaN,-0.034736,0.021914,NaN,NaN,NaN,NaN,...,NaN,NaN,-0.038323,NaN,29.000000,1.250567e-07,11.000000,-2.296315,-2.318077,0.018519
1,10001,1990-02-01,1.839530,NaN,-0.034736,0.021914,NaN,NaN,NaN,NaN,...,NaN,NaN,-0.038516,NaN,27.889328,1.909091e+00,7.875000,-2.290006,-2.311768,0.006289
2,10001,1990-03-01,1.830574,NaN,-0.034736,0.021914,NaN,NaN,NaN,NaN,...,NaN,NaN,-0.038655,NaN,29.881423,4.421053e+00,10.161291,-2.290006,-2.316648,-0.012658
3,10001,1990-04-01,1.830574,NaN,-0.034736,0.021914,NaN,NaN,NaN,NaN,...,NaN,NaN,-0.038695,NaN,32.869565,4.772727e+00,14.000000,-2.290006,-2.316648,-0.000000
4,10001,1990-05-01,1.854043,NaN,-0.034736,0.021914,NaN,NaN,NaN,NaN,...,NaN,NaN,-0.038640,NaN,40.837945,9.450000e+00,22.354839,-2.277267,-2.303909,0.012658
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3476088,93436,2025-09-01,NaN,NaN,-0.013895,0.032673,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.692751,NaN,NaN,NaN,NaN,NaN,NaN
3476089,93436,2025-10-01,NaN,NaN,-0.013895,0.032673,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.692751,NaN,NaN,NaN,NaN,NaN,NaN
3476090,93436,2025-11-01,NaN,NaN,-0.013895,0.032673,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.692751,NaN,NaN,NaN,NaN,NaN,NaN
3476091,93436,2025-12-01,NaN,NaN,-0.013895,0.032673,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.692751,NaN,NaN,NaN,NaN,NaN,NaN
